# 01 - M1 gate on the real data
Run top to bottom (CPU is fine, no GPU needed). The first cell is the shared bootstrap.
Checks: held-out generator absent from train/val, splits balanced, manifests match the pinned dataset revision, controlled protocol applies exactly one JPEG q90.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'main'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
!python -m pytest -q

In [ ]:
# Load the dataset at the revision pinned in the manifests
import numpy as np
from datasets import load_dataset
from src.data import load_manifest, audit_manifest
FOLDS = ['Midjourney', 'BigGAN', 'ADM']
rev = load_manifest('Midjourney')['revision']
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=rev)
names = ds['train'].features['generator'].names
train_gen = np.asarray(ds['train']['generator']); val_gen = np.asarray(ds['validation']['generator'])
print('revision', rev, '\nnames', names)

In [ ]:
# Gate 1+2: leakage, balance, test 500/500 - fails loudly on any violation
for f in FOLDS:
    m = load_manifest(f)
    assert m['revision'] == rev
    print(f, audit_manifest(m, train_gen, val_gen, names))
# determinism: manifests are fixed files, so re-loading gives identical lists
assert load_manifest('ADM')['train'] == load_manifest('ADM')['train']
print('M1 gate: leakage/balance OK')

In [ ]:
# Gate 3: controlled protocol on real images from every class: one JPEG, 256 px, q90 tables
import io
from PIL import Image
from src.data import controlled_pre, preprocess
rows = []
for g in range(len(names)):
    idx = np.where(train_gen == g)[0][:5]
    for i in idx:
        im = ds['train'][int(i)]['image']
        out = controlled_pre(im)
        assert out.size == (256, 256)
        x = preprocess(im, 'controlled'); assert tuple(x.shape) == (3, 336, 336)
        rows.append((names[g], im.format, im.size))
print(rows[:3], '... checked', len(rows), 'images across', len(set(r[0] for r in rows)), 'classes')
print('controlled protocol OK')